# Task 2: End-to-End ML Pipeline with Scikit-learn
**DevelopersHub Corporation – AI/ML Engineering Advanced Internship**

---

## Problem Statement & Objective
Build a **reusable, production-ready ML pipeline** to predict customer churn using the **Telco Customer Churn Dataset**.

**Churn** = a customer cancels their subscription. Predicting it allows businesses to take proactive retention actions.

## Approach
1. Load and explore the Telco Churn dataset
2. Build preprocessing pipelines (StandardScaler + OneHotEncoder via ColumnTransformer)
3. Train **Logistic Regression** and **Random Forest** inside sklearn `Pipeline`
4. Tune hyperparameters with **GridSearchCV** (5-fold CV) — display full results table
5. Evaluate using Accuracy, F1, ROC-AUC, confusion matrix, ROC curve
6. Export best pipeline using **joblib**

In [ ]:
# Step 1: Install libraries
!pip install pandas numpy scikit-learn matplotlib seaborn joblib -q

In [ ]:
# Step 2: Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix,
    roc_curve, ConfusionMatrixDisplay
)
print('Libraries ready.')

In [ ]:
# Step 3: Dataset Loading
# IBM Telco Customer Churn Dataset (public)
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# Dataset overview
print('=== Data Types ===')
print(df.dtypes.to_string())
print(f'\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}')
print(f'\nChurn distribution:\n{df["Churn"].value_counts()}')
print(f'Churn Rate: {(df["Churn"]=="Yes").mean()*100:.1f}%')

In [ ]:
# Visualization 1: EDA Overview (4 plots)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Churn distribution
churn_counts = df['Churn'].value_counts()
axes[0, 0].pie(churn_counts, labels=['No Churn', 'Churn'], autopct='%1.1f%%',
               colors=['#4C72B0', '#DD8452'], startangle=140, explode=[0, 0.05])
axes[0, 0].set_title('Churn Distribution', fontweight='bold')

# Churn by Contract
contract_churn = df.groupby(['Contract', 'Churn']).size().unstack(fill_value=0)
contract_churn.plot(kind='bar', ax=axes[0, 1], color=['#4C72B0', '#DD8452'], edgecolor='white')
axes[0, 1].set_title('Churn by Contract Type', fontweight='bold')
axes[0, 1].tick_params(axis='x', rotation=0)

# Tenure distribution by churn
df[df['Churn']=='No']['tenure'].hist(ax=axes[1, 0], bins=30, alpha=0.6, color='#4C72B0', label='No Churn')
df[df['Churn']=='Yes']['tenure'].hist(ax=axes[1, 0], bins=30, alpha=0.6, color='#DD8452', label='Churn')
axes[1, 0].set_title('Tenure Distribution by Churn', fontweight='bold')
axes[1, 0].set_xlabel('Tenure (months)')
axes[1, 0].legend()

# Monthly Charges by churn
df.boxplot(column='MonthlyCharges', by='Churn', ax=axes[1, 1],
           boxprops=dict(color='#4C72B0'), medianprops=dict(color='red'))
axes[1, 1].set_title('Monthly Charges by Churn', fontweight='bold')
axes[1, 1].set_xlabel('Churn')
plt.suptitle('')

plt.tight_layout()
plt.savefig('eda_churn.png', dpi=150)
plt.show()

In [ ]:
# Step 4: Data Preprocessing
df = df.drop(columns=['customerID'])  # Not a useful feature
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)
df['Churn'] = (df['Churn'] == 'Yes').astype(int)  # Binary encode: Yes->1, No->0

X = df.drop(columns=['Churn'])
y = df['Churn']

numerical_cols   = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

print(f'Numerical features  ({len(numerical_cols)}): {numerical_cols}')
print(f'Categorical features ({len(categorical_cols)}): {categorical_cols}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

In [ ]:
# Step 5: Build Scikit-learn Pipelines
# ColumnTransformer applies different preprocessing per column type

preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([('scaler', StandardScaler())]), numerical_cols),
    ('cat', Pipeline([('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical_cols)
])

# Pipeline 1: Logistic Regression
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# Pipeline 2: Random Forest
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42, n_jobs=-1))
])

print('Pipelines created:')
print('  Pipeline 1: Preprocessor → Logistic Regression')
print('  Pipeline 2: Preprocessor → Random Forest')

In [ ]:
# Step 6a: GridSearchCV – Logistic Regression
# C = inverse regularization strength (smaller = stronger regularization)
lr_param_grid = {
    'classifier__C': [0.01, 0.1, 1.0, 10.0],
    'classifier__solver': ['liblinear', 'lbfgs']
}

lr_grid = GridSearchCV(
    lr_pipeline, lr_param_grid,
    cv=5, scoring='f1', n_jobs=-1, verbose=1
)
lr_grid.fit(X_train, y_train)

print(f'\nBest LR parameters : {lr_grid.best_params_}')
print(f'Best CV F1 Score   : {lr_grid.best_score_:.4f}')

# Show full GridSearch results table
lr_results = pd.DataFrame(lr_grid.cv_results_)
lr_table = lr_results[['param_classifier__C', 'param_classifier__solver',
                        'mean_test_score', 'std_test_score', 'rank_test_score']]
lr_table.columns = ['C', 'Solver', 'Mean F1', 'Std F1', 'Rank']
lr_table = lr_table.sort_values('Rank')
print('\nGridSearch Results (Logistic Regression):')
print(lr_table.to_string(index=False))

In [ ]:
# Step 6b: GridSearchCV – Random Forest
rf_param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [None, 5, 10],
    'classifier__min_samples_split': [2, 5]
}

rf_grid = GridSearchCV(
    rf_pipeline, rf_param_grid,
    cv=5, scoring='f1', n_jobs=-1, verbose=1
)
rf_grid.fit(X_train, y_train)

print(f'\nBest RF parameters : {rf_grid.best_params_}')
print(f'Best CV F1 Score   : {rf_grid.best_score_:.4f}')

# Full GridSearch results
rf_results = pd.DataFrame(rf_grid.cv_results_)
rf_table = rf_results[['param_classifier__n_estimators', 'param_classifier__max_depth',
                        'param_classifier__min_samples_split',
                        'mean_test_score', 'std_test_score', 'rank_test_score']]
rf_table.columns = ['n_estimators', 'max_depth', 'min_samples_split', 'Mean F1', 'Std F1', 'Rank']
rf_table = rf_table.sort_values('Rank')
print('\nTop 5 GridSearch Results (Random Forest):')
print(rf_table.head(5).to_string(index=False))

In [ ]:
# Visualization 2: GridSearch Heatmap (Random Forest – n_estimators vs max_depth)
pivot = rf_results.pivot_table(
    values='mean_test_score',
    index='param_classifier__max_depth',
    columns='param_classifier__n_estimators'
)
plt.figure(figsize=(8, 4))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlGnBu')
plt.title('GridSearch F1 Scores – Random Forest\n(n_estimators vs max_depth)', fontweight='bold')
plt.xlabel('n_estimators')
plt.ylabel('max_depth')
plt.tight_layout()
plt.savefig('gridsearch_heatmap.png', dpi=150)
plt.show()

In [ ]:
# Step 6c: 5-Fold Cross-Validation Scores
print('=== 5-Fold Cross-Validation on Training Set ===')
for name, estimator in [('Logistic Regression', lr_grid.best_estimator_),
                        ('Random Forest',       rf_grid.best_estimator_)]:
    cv_scores = cross_val_score(estimator, X_train, y_train, cv=5, scoring='f1')
    print(f'{name:<25} CV F1: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
    print(f'                          Fold scores: {[round(s,4) for s in cv_scores]}')

In [ ]:
# Step 7: Evaluate Both Models on Test Set
def evaluate_model(model, X_test, y_test, name):
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    acc  = accuracy_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)
    auc  = roc_auc_score(y_test, y_proba)
    print(f'\n{"="*42}')
    print(f'  {name}')
    print(f'{"="*42}')
    print(f'  Accuracy  : {acc:.4f}')
    print(f'  F1 Score  : {f1:.4f}')
    print(f'  ROC-AUC   : {auc:.4f}')
    print('\nClassification Report:')
    print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))
    return y_pred, y_proba, {'Accuracy': acc, 'F1': f1, 'ROC-AUC': auc}

lr_preds, lr_proba, lr_m = evaluate_model(lr_grid, X_test, y_test, 'Logistic Regression')
rf_preds, rf_proba, rf_m = evaluate_model(rf_grid, X_test, y_test, 'Random Forest')

In [ ]:
# Visualization 3: Model Comparison + ROC Curves + Confusion Matrix
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# 1. Metrics comparison bar chart
metrics_df = pd.DataFrame([lr_m, rf_m], index=['Logistic Regression', 'Random Forest'])
metrics_df.plot(kind='bar', ax=axes[0], color=['#4C72B0', '#DD8452', '#55A868'], edgecolor='white')
axes[0].set_title('Model Metrics Comparison', fontweight='bold')
axes[0].set_ylim(0, 1.15)
axes[0].tick_params(axis='x', rotation=15)
axes[0].legend(loc='lower right')
for container in axes[0].containers:
    axes[0].bar_label(container, fmt='%.2f', padding=2, fontsize=8)

# 2. ROC Curves
for proba, name, color in [(lr_proba,'LR','#4C72B0'), (rf_proba,'RF','#DD8452')]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    axes[1].plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, linewidth=2)
axes[1].plot([0,1],[0,1],'k--', alpha=0.4, label='Random')
axes[1].set_title('ROC Curve Comparison', fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()

# 3. Confusion Matrix (best model = RF)
cm = confusion_matrix(y_test, rf_preds)
ConfusionMatrixDisplay(cm, display_labels=['No Churn','Churn']).plot(
    ax=axes[2], cmap='Blues', colorbar=False)
axes[2].set_title('Random Forest Confusion Matrix', fontweight='bold')

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150)
plt.show()

In [ ]:
# Visualization 4: Feature Importances (Random Forest)
best_rf    = rf_grid.best_estimator_
ohe_feats  = best_rf.named_steps['preprocessor'] \
    .named_transformers_['cat'] \
    .named_steps['encoder'] \
    .get_feature_names_out(categorical_cols)
all_feats  = numerical_cols + list(ohe_feats)
importances = best_rf.named_steps['classifier'].feature_importances_

feat_df = pd.DataFrame({'Feature': all_feats, 'Importance': importances})
feat_df = feat_df.sort_values('Importance', ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(data=feat_df, x='Importance', y='Feature', palette='viridis')
plt.title('Top 15 Feature Importances – Random Forest', fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

In [ ]:
# Step 8: Export Pipelines with joblib
joblib.dump(rf_grid.best_estimator_, 'churn_pipeline_rf.joblib')
joblib.dump(lr_grid.best_estimator_, 'churn_pipeline_lr.joblib')
print('Saved: churn_pipeline_rf.joblib')
print('Saved: churn_pipeline_lr.joblib')

# Reload and use for prediction
loaded_pipeline = joblib.load('churn_pipeline_rf.joblib')
sample    = X_test.iloc[[0]]
pred      = loaded_pipeline.predict(sample)[0]
prob      = loaded_pipeline.predict_proba(sample)[0][1]
print(f'\nSample prediction from reloaded pipeline:')
print(f'  Churn Prediction  : {"Yes" if pred == 1 else "No"}')
print(f'  Churn Probability : {prob:.2%}')

## Final Summary & Insights

### Pipeline Architecture
```
Raw Data (21 features)
      ↓
ColumnTransformer
  ├─ Numerical (7 cols)  → StandardScaler  → scaled features
  └─ Categorical (14 cols) → OneHotEncoder → binary features
      ↓
Classifier (LR or Random Forest)
      ↓
Prediction (Churn=1 / No Churn=0) + Probability
```

### Key Results
| Model | Accuracy | F1 | ROC-AUC |
|-------|----------|----|---------|
| Logistic Regression | ~80% | ~0.57 | ~0.85 |
| Random Forest | ~80–82% | ~0.60 | ~0.86 |

### Key Insights
- **Tenure** is the single strongest churn predictor — long-term customers rarely churn
- **Month-to-month contract** customers churn at the highest rate (~43%)
- **Electronic check** payment method correlates with churn
- The dataset is imbalanced (~26% churn) — F1 is a more reliable metric than Accuracy
- Random Forest outperforms LR on F1 due to its ability to capture non-linear patterns

### Skills Demonstrated
- ML pipeline construction with sklearn `Pipeline` and `ColumnTransformer`
- Hyperparameter tuning with `GridSearchCV` (5-fold CV)
- Model export and reuse with `joblib`
- Production-readiness: end-to-end preprocessing included in the exported pipeline